# Does a transformer earn its cost on 20k documents?

20 Newsgroups is small by modern standards: roughly 20,000 Usenet messages across 20
categories, of which this experiment uses fifteen. Small enough that the interesting
question is not "can a transformer classify this" — it can — but **how much it buys over
a dense baseline, and what that gain costs in parameters and training time.**

Both models are trained under identical conditions: same vocabulary, same sequence
length, same split, same seed, same loss, same optimizer, same number of epochs. That is
enforced by a single immutable `ModelConfig` rather than by discipline across notebook
cells.

Everything below imports `src/newsgroups/`, which is covered by the test suite.

## Setup

```bash
pip install -e ".[dev]"
pip install -r requirements.txt   # TensorFlow
```

See the README for how to download the dataset.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

from newsgroups import truncation
from newsgroups.corpus import (
    CATEGORIES,
    counts_by_category,
    iter_texts,
    load_corpus,
    strip_headers,
)
from newsgroups.experiment import compare, run
from newsgroups.models import ModelConfig
from newsgroups.split import train_validation_split

DATA_DIR = Path.home() / ".keras/datasets/news20_extracted/20_newsgroup"

## The decision that determines whether any number here means anything

Every message begins with an RFC-822 header block, and one of those headers is
`Newsgroups: comp.graphics` — the label. Leave it in the training text and the model
scores beautifully by reading the answer off its own input.

The usual fix in circulation is `lines[10:]`: drop ten lines and hope. That is wrong in
both directions and silent in both. A message with fewer than ten header lines loses the
start of its body; a message with more keeps the label.

`strip_headers` splits on the blank line that actually ends the header block, and drops
any leaking header that survives an unconventional layout.

In [ ]:
with open(DATA_DIR / "comp.graphics" / "37261", encoding="latin-1") as handle:
    raw = handle.read()
print(raw[:400])
print("\n--- after stripping ---\n")
print(strip_headers(raw)[:400])

In [ ]:
# examples/leakage_demo.py quantifies exactly what that is worth, with no dataset
# and no TensorFlow: a classifier that only searches for the category name scores
# 1.000 on the leaking corpus and chance on the stripped one.
!python examples/leakage_demo.py

## Loading and splitting

The split shuffles `Document` objects rather than a text list and a label list in two
separate calls. The two-list version works only because both calls re-seed the same
generator; one added filter, one changed seed, and every label belongs to a different
document — with nothing raising and accuracy simply sitting near chance.

In [ ]:
documents = load_corpus(DATA_DIR, CATEGORIES)
split = train_validation_split(documents, validation_fraction=0.2, seed=1337)

print(f"documents {len(documents):,}   train/val {split.sizes[0]:,}/{split.sizes[1]:,}")
for category, count in sorted(counts_by_category(documents).items()):
    print(f"  {category:<26}{count:>6}")

## What does a 200-token sequence actually cover?

`output_sequence_length=200` is a tutorial constant that quietly decides how much of each
message the model is allowed to read. Two different numbers are worth having, because they
can point in opposite directions: the share of **documents** kept whole, and the share of
**all words** kept.

In [ ]:
for length in (100, 200, 400, 1000):
    print(truncation.analyse(iter_texts(documents), length).summary())
    print()

## Training both architectures

`run()` adapts the vectorizer on the **training split only** — adapting on the full corpus
leaks validation vocabulary into training — then builds, compiles and fits one
architecture, and scores it on the validation half.

Two details in `models.py` that the common version of this notebook gets wrong:

- the transformer block forwards `training` instead of hardcoding `training=False`, which
  would disable both dropout layers for the entire run;
- the output layer is sized from `ModelConfig.n_classes`, not from a literal `20` while
  training on fifteen categories.

In [ ]:
config = ModelConfig(
    n_classes=len(CATEGORIES), sequence_length=200, vocab_size=20_000, seed=1337
)

results = [
    run(split, config, architecture="dense", epochs=20),
    run(split, config, architecture="transformer", epochs=20),
]

## Results

Accuracy alone would not settle this. The classes are close to balanced, so accuracy is
not meaningless — but the two models can tie on it while differing sharply on the small
overlapping categories (`talk.religion.misc` against `alt.atheism`). Macro-F1 weights
every class equally and makes that visible.

Each accuracy is printed against the majority-class baseline, which on fifteen roughly
balanced classes is about 0.067.

In [ ]:
for result in results:
    print(
        f"=== {result.name}  (baseline {result.baseline_accuracy:.3f}, "
        f"lift {result.lift_over_baseline:+.3f})"
    )
    print(result.report.summary())
    print()

In [ ]:
print(compare(results))

In [ ]:
# The three classes each model handles worst — usually the semantically
# overlapping pairs, and the place where the two architectures differ most.
for result in results:
    worst = ", ".join(f"{s.name} ({s.f1:.2f})" for s in result.report.worst(3))
    print(f"{result.name:<14}{worst}")

## Conclusion

The number to carry out of this is not "which model won" but the **shape of the
trade-off**: the macro-F1 gap against the parameter and wall-clock cost of closing it.
On a corpus this size, a transformer trained from scratch is competing against a dense
network on data far below the scale where attention pays for itself, and a result either
way is informative.

What would change the answer, and is deliberately out of scope here:

- pre-trained embeddings or a fine-tuned encoder — a different experiment with a foregone
  conclusion;
- per-model hyperparameter search — tuning one side and not the other is the usual way
  this comparison gets its result;
- repeated seeds — one run is enough to show the cost shape, not enough to defend a
  half-point gap.